# EDA — Credit Risk Scoring

Análise exploratória dos dados de risco de crédito. Objetivo: entender o perfil de inadimplência (default) e levantar hipóteses de negócio que guiam a modelagem.

Rode o ETL antes: `python -m src.data.etl`

In [ ]:
import sys; sys.path.append('..')
import pandas as pd
from src import config

df = pd.read_parquet(config.PROCESSED_PARQUET) if config.PROCESSED_PARQUET.exists() else pd.read_csv(config.RAW_CSV)
print(df.shape)
df.head()

## Taxa de default (desbalanceamento)

In [ ]:
df['default'].value_counts(normalize=True).rename('proporcao')

A classe positiva (default) é minoritária (~20%). Isso justifica **SMOTE** no treino e o uso de métricas como PR-AUC/recall em vez de acurácia.

## Default por segmento

In [ ]:
for col in ['home_ownership', 'loan_intent']:
    display(df.groupby(col)['default'].mean().sort_values(ascending=False))

## Correlação das variáveis numéricas com o default

In [ ]:
num = df.select_dtypes('number')
num.corr()['default'].sort_values(ascending=False)

### Insights de negócio

- **Comprometimento de renda** (`loan_percent_income`) e **endividamento** (`debt_to_income`) são os maiores fatores de risco.
- **Atrasos passados** (`past_delinquencies`) elevam fortemente a probabilidade de default.
- **Renda** e **histórico de crédito** longos reduzem o risco.
- Quem mora de **aluguel** tende a inadimplir um pouco mais do que quem tem imóvel próprio/financiado.

Esses sinais orientam a feature engineering (Etapa 4) e são confirmados pelo SHAP (Etapa 5).